### __Data Cleaning__

In [2]:
# Required Dependencies 
import pandas as pd 
import numpy as np

In [ ]:
# SUMMARY TABLE

data = pd.read_csv('NY-House-Dataset.csv/NY-House-Dataset.csv') 
data.head(5)



,BROKERTITLE,TYPE,PRICE,BEDS,BATH,PROPERTYSQFT,ADDRESS,STATE,MAIN_ADDRESS,ADMINISTRATIVE_AREA_LEVEL_2,LOCALITY,SUBLOCALITY,STREET_NAME,LONG_NAME,FORMATTED_ADDRESS,LATITUDE,LONGITUDE
0,Brokered by Douglas Elliman -111 Fifth Ave,Condo for sale,315000,2,2.000000,1400.0,2 E 55th St Unit 803,"New York, NY 10022","2 E 55th St Unit 803New York, NY 10022",New York County,New York,Manhattan,East 55th Street,Regis Residence,"Regis Residence, 2 E 55th St #803, New York, N...",40.761255,-73.974483
1,Brokered by Serhant,Condo for sale,195000000,7,10.000000,17545.0,Central Park Tower Penthouse-217 W 57th New Yo...,"New York, NY 10019",Central Park Tower Penthouse-217 W 57th New Yo...,United States,New York,New York County,New York,West 57th Street,"217 W 57th St, New York, NY 10019, USA",40.766393,-73.980991
2,Brokered by Sowae Corp,House for sale,260000,4,2.000000,2015.0,620 Sinclair Ave,"Staten Island, NY 10312","620 Sinclair AveStaten Island, NY 10312",United States,New York,Richmond County,Staten Island,Sinclair Avenue,"620 Sinclair Ave, Staten Island, NY 10312, USA",40.541805,-74.196109
3,Brokered by COMPASS,Condo for sale,69000,3,1.000000,445.0,2 E 55th St Unit 908W33,"Manhattan, NY 10022","2 E 55th St Unit 908W33Manhattan, NY 10022",United States,New York,New York County,New York,East 55th Street,"2 E 55th St, New York, NY 10022, USA",40.761398,-73.974613
4,Brokered by Sotheby's International Realty - E...,Townhouse for sale,55000000,7,2.373861,14175.0,5 E 64th St,"New York, NY 10065","5 E 64th StNew York, NY 10065",United States,New York,New York County,New York,East 64th Street,"5 E 64th St, New York, NY 10065, USA",40.767224,-73.969856


In [ ]:
# ============================================================
# 0) SETUP
# ============================================================
# pip install pandas numpy scikit-learn

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ------------------------------------------------------------
# 1) LOAD DATA
# ------------------------------------------------------------
# Cambia la ruta a tu archivo
df = pd.read_csv("New_York_Housing_Market.csv")

# ------------------------------------------------------------
# 2) BASIC CLEANING (ajusta nombres según tu CSV)
# ------------------------------------------------------------
# Normaliza nombres de columnas (opcional)
df.columns = [c.strip().upper().replace(" ", "_") for c in df.columns]

# Columnas típicas del dataset (ajusta si difieren)
target = "PRICE"

# IMPORTANT: asegúrate de que LATITUDE y LONGITUDE existan en tu archivo
# (si no, reemplaza por el nombre correcto)
geo_cols = ["LATITUDE", "LONGITUDE"]

# Features base (modelo hedónico simple)
# Puedes quitar/añadir según tu dataset
candidate_features = [
    "BEDS", "BATH", "PROPERTYSQFT", "TYPE",
    "ADMINISTRATIVE_AREA_LEVEL_2", "LOCALITY", "SUBLOCALITY",
    "BROKERTITLE"
] + geo_cols

# Filtra a solo columnas existentes en el df (por si hay diferencias)
features = [c for c in candidate_features if c in df.columns]

# Subset y limpieza mínima
data = df[features + [target]].copy()

# PRICE numérico (por si viene como texto con símbolos)
data[target] = (
    data[target]
    .astype(str)
    .str.replace(r"[^0-9.\-]", "", regex=True)
    .replace("", np.nan)
    .astype(float)
)

# Elimina precios faltantes o no positivos (log requiere >0)
data = data.dropna(subset=[target])
data = data[data[target] > 0].copy()

# Transformación recomendada para colas largas de precios
data["LOG_PRICE"] = np.log(data[target])

X = data[features]
y = data["LOG_PRICE"]

# ------------------------------------------------------------
# 3) TRAIN / TEST SPLIT
# ------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ------------------------------------------------------------
# 4) PREPROCESSING (numéricas vs categóricas)
# ------------------------------------------------------------
numeric_features = [c for c in features if c in ["BEDS", "BATH", "PROPERTYSQFT", "LATITUDE", "LONGITUDE"]]
categorical_features = [c for c in features if c not in numeric_features]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=10))
    # min_frequency agrupa categorías raras; ajusta (p.ej. 5, 20) según tu tamaño
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

# ------------------------------------------------------------
# 5) BASELINE MODELS: Linear, Ridge, Lasso
# ------------------------------------------------------------
models = {
    "LinearRegression": LinearRegression(),
    "Ridge(alpha=1.0)": Ridge(alpha=1.0, random_state=42),
    "Lasso(alpha=0.001)": Lasso(alpha=0.001, random_state=42, max_iter=5000),
}

def evaluate_log_model(name, pipe, X_test, y_test):
    yhat_log = pipe.predict(X_test)

    # Métricas en escala log (útiles para entrenamiento)
    rmse_log = mean_squared_error(y_test, yhat_log, squared=False)
    mae_log = mean_absolute_error(y_test, yhat_log)
    r2 = r2_score(y_test, yhat_log)

    # Métricas aproximadas en escala original ($)
    # Nota: si quieres corrección por sesgo (smearing), se hace aparte.
    y_true = np.exp(y_test)
    y_pred = np.exp(yhat_log)

    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mae = mean_absolute_error(y_true, y_pred)

    return {
        "model": name,
        "RMSE_log": rmse_log,
        "MAE_log": mae_log,
        "R2_log": r2,
        "RMSE_$": rmse,
        "MAE_$": mae,
    }

results = []
fitted = {}

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    results.append(evaluate_log_model(name, pipe, X_test, y_test))

results_df = pd.DataFrame(results).sort_values("RMSE_log")
print(results_df)

best_name = results_df.iloc[0]["model"]
best_model = fitted[best_name]
print("\nBest model:", best_name)

# ------------------------------------------------------------
# 6) RESIDUALS + EXPORT FOR GEODA/QGIS
# ------------------------------------------------------------
# Residuals en log y en escala original
yhat_test_log = best_model.predict(X_test)
resid_log = (y_test - yhat_test_log)

# Precios reales / predichos (aprox) en $
y_true = np.exp(y_test)
y_pred = np.exp(yhat_test_log)

out = X_test.copy()
out["PRICE_TRUE"] = y_true.values
out["PRICE_PRED"] = y_pred.values
out["RESID_LOG"] = resid_log.values
out["RESID_$"] = (y_true.values - y_pred.values)

# Guarda a CSV para llevar a QGIS/GeoDa (puntos con LAT/LON)
# IMPORTANTE: GeoDa trabaja mejor con un ID único
out = out.reset_index(drop=True)
out.insert(0, "ID", np.arange(1, len(out) + 1))

out.to_csv("geoda_residuals_points.csv", index=False)
print("\nSaved: geoda_residuals_points.csv")

# (Opcional) Resumen rápido del error absoluto porcentual
mape = np.mean(np.abs((out["PRICE_TRUE"] - out["PRICE_PRED"]) / out["PRICE_TRUE"])) * 100
print(f"MAPE (%): {mape:.2f}")
